In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = (
    os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]
)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:
import pyspark.sql as ps
import pyspark.sql.functions as F 
import pyspark.sql.types as pst
import pandas as pdS
import pyspark.sql.window as psw
from pyspark.sql import SparkSession


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EV practice")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print(spark)
print("Spark Version:", spark.version)

c:\Users\chouh\Desktop\data engineer\Notes\Pyspark_practice\de_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Version: 4.2.0


In [5]:
file_path = r"C:\Users\chouh\Desktop\data engineer\Notes\coingenco_ETL_Project\input\coingecko_raw_data.csv"

print(os.path.exists(file_path))


True


In [6]:
df = ( spark
    .read
    .option("samplingRatio",0.001)
    .option("header" , True)
    .option("inferSchema",True)
    .csv(file_path)
)

In [7]:
print(os.listdir())

['check.py', 'coingecko_practice.ipynb', 'input', 'Kafka', 'output', 'Pyspark_ETL.py', 'venv']


In [8]:
df.show(5)

+-----------+------+--------+--------------------+-------------+-------------+---------------+-----------------------+-------------+--------+--------+-------------------+---------------------------+---------------------+--------------------------------+------------------+-----------------+-----------+--------+---------------------+--------------------+---------+---------------------+--------------------+--------------------+--------------------+
|         id|symbol|    name|               image|current_price|   market_cap|market_cap_rank|fully_diluted_valuation| total_volume|high_24h| low_24h|   price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|     total_supply| max_supply|     ath|ath_change_percentage|            ath_date|      atl|atl_change_percentage|            atl_date|                 roi|        last_updated|
+-----------+------+--------+--------------------+-------------+-------------+---------------+------

In [9]:
print(20068728.0*64407.0)

1292566564296.0


In [10]:
print("Total_Rows :-"  , df.count())
print("Total_columns :-" , len(df.columns))

Total_Rows :- 250
Total_columns :- 26


In [11]:
df.filter(F.col("max_supply").isNull()).count()

104

In [12]:
# this is not null values
df.filter(F.col("max_supply").isNotNull()).count()

146

In [13]:
# cast correct data type in each column.
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- image: string (nullable = true)
 |-- current_price: string (nullable = true)
 |-- market_cap: string (nullable = true)
 |-- market_cap_rank: string (nullable = true)
 |-- fully_diluted_valuation: string (nullable = true)
 |-- total_volume: string (nullable = true)
 |-- high_24h: string (nullable = true)
 |-- low_24h: string (nullable = true)
 |-- price_change_24h: string (nullable = true)
 |-- price_change_percentage_24h: string (nullable = true)
 |-- market_cap_change_24h: string (nullable = true)
 |-- market_cap_change_percentage_24h: string (nullable = true)
 |-- circulating_supply: string (nullable = true)
 |-- total_supply: string (nullable = true)
 |-- max_supply: string (nullable = true)
 |-- ath: string (nullable = true)
 |-- ath_change_percentage: string (nullable = true)
 |-- ath_date: string (nullable = true)
 |-- atl: string (nullable = true)
 |-- atl_change

In [14]:
# convert string values into double
double_columns = [
    "current_price",
    "total_volume",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "circulating_supply",
    "total_supply",
    "max_supply",
    "ath",
    "atl"
]

for c in double_columns:
    df = df.withColumn(c , F.col(c).cast("double"))


In [15]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- image: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: string (nullable = true)
 |-- market_cap_rank: string (nullable = true)
 |-- fully_diluted_valuation: string (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- market_cap_change_24h: string (nullable = true)
 |-- market_cap_change_percentage_24h: string (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- total_supply: double (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: string (nullable = true)
 |-- ath_date: string (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change

In [16]:
# Convert integer and long columns separately:
df = df.withColumn(
    "market_cap_rank",
    F.col("market_cap_rank").cast("integer")
)

df = df.withColumn(
    "market_cap",
    F.col("market_cap").cast("long")
)

In [17]:
# Find missing values per column 
df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df.columns
]).show()

+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+---------------------+--------+---+------------+
| id|symbol|name|image|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|ath_date|atl|atl_change_percentage|atl_date|roi|last_updated|
+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+--

In [18]:
numeric_cols = [
    "price_change_24h",
    "market_cap_change_24h",
    "market_cap_change_percentage_24h"
]

for c in numeric_cols:
    df = df.withColumn(
        c,
        F.col(c).cast("double")
    )

In [19]:
# Fill Null values
# inputCols → Jin columns ke NULL fill karne hain.
# outputCols → Output mein columns ke names.
# setStrategy("median") → Median value se NULL replace karega.
# fit(df) → Har column ka median calculate karega.
# transform(df) → NULL values fill karega.

from pyspark.ml.feature import Imputer

cols = [
    "price_change_24h",
    "market_cap_change_24h",
    "market_cap_change_percentage_24h"
]

imputer = Imputer(
    inputCols=cols,
    outputCols=cols
).setStrategy("median")

df = imputer.fit(df).transform(df)

In [20]:
# FIND DUPLICATES
df.groupBy("id").count().filter(F.col("count")>1).show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [21]:
df.columns

['id',
 'symbol',
 'name',
 'image',
 'current_price',
 'market_cap',
 'market_cap_rank',
 'fully_diluted_valuation',
 'total_volume',
 'high_24h',
 'low_24h',
 'price_change_24h',
 'price_change_percentage_24h',
 'market_cap_change_24h',
 'market_cap_change_percentage_24h',
 'circulating_supply',
 'total_supply',
 'max_supply',
 'ath',
 'ath_change_percentage',
 'ath_date',
 'atl',
 'atl_change_percentage',
 'atl_date',
 'roi',
 'last_updated']

In [22]:
df.orderBy(F.col("market_cap").desc()).select("market_cap","id","symbol").show(5)
df.select("market_cap","id","symbol").orderBy(F.desc(F.col("market_cap"))).show(5)

+-------------+-----------+------+
|   market_cap|         id|symbol|
+-------------+-----------+------+
|1292375743620|    bitcoin|   btc|
| 228584712931|   ethereum|   eth|
| 183100236344|     tether|  usdt|
|  81307986747|binancecoin|   bnb|
|  72097386899|   usd-coin|  usdc|
+-------------+-----------+------+
only showing top 5 rows
+-------------+-----------+------+
|   market_cap|         id|symbol|
+-------------+-----------+------+
|1292375743620|    bitcoin|   btc|
| 228584712931|   ethereum|   eth|
| 183100236344|     tether|  usdt|
|  81307986747|binancecoin|   bnb|
|  72097386899|   usd-coin|  usdc|
+-------------+-----------+------+
only showing top 5 rows


In [25]:
df.groupBy(df.columns).count()\
.filter(F.col("count")>1).show()

+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+---------------------+--------+---+------------+-----+
| id|symbol|name|image|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|ath_date|atl|atl_change_percentage|atl_date|roi|last_updated|count|
+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+---

In [33]:
df.dropDuplicates().count()

250